# SYNPRED on BreastDCEDL: demo

This notebook runs the full SYNPRED pipeline on the BreastDCEDL / I-SPY2 data:

1. **Load the data** (2-channel T0 MIPs plus I-SPY2 gene expression) and show samples from the dataloader.
2. **Get a backbone**: either pretrain SYNPRED from scratch (`RUN_PRETRAINING = True`) or load released checkpoints.
3. **Frozen evaluation**: run a linear probe on the task-relevant embeddings (`[z_fusion ; z_uni]`, combined across modalities with a product of experts). The probe is trained on multimodal embeddings and tested on IMG-only, RNA-only and multimodal inputs. It is run on the checkpoint selected by validation AUROC and on the last-epoch checkpoint.
4. **Fine-tuning**: train an MLP head on top of the backbone (frozen by default).
5. **Predictions**: show test images with their true and predicted labels, plus ROC curves.
6. **All seeds**: summarise the linear probe across the five pretraining repetitions (last-epoch checkpoints).

Set the paths in the next cell. Everything else runs as is. Hyperparameters come from `configs/synpred.yaml`.

Each step has a command-line equivalent: `scripts/pretrain.py`, `scripts/evaluate.py`, `scripts/finetune.py` and `scripts/summarize.py`. See the README.

## 0. Configuration

In [ ]:
from pathlib import Path

DATA_PATH = "/home/ivana/synergy_mm/data/BC_pcr_ds"   # folder with train.pkl / val.pkl / test.pkl
RNA_PATH = "/home/ivana/SYNPRED/rna/GSE194040_ISPY2ResID_AgilentGeneExp_990_FrshFrzn_meanCol_geneLevel_n988.txt"
# Released backbones, one folder per pretraining repetition (rep_0 ... rep_4):
#   model_best_auc.pt  selected by validation AUROC (multimodal linear probe, epoch >= 50)
#   model.pt           last epoch (scripts/pretrain.py calls it model_last.pt)
RELEASED_DIR = Path("/home/ivana/SYNPRED/pretrained_models/BreastDCEDL_res/ablations/ablation_all/"
                    "beta_0.0001/lr_1e-05_1e-06/cls_dim_16")
REPS = range(5)
# The first entry of each list is used for sections 3-6; section 7 loops over LAST_CHECKPOINT_PATHS.
CHECKPOINT_PATHS = [str(RELEASED_DIR / f"rep_{r}" / "model_best_auc.pt") for r in REPS]
LAST_CHECKPOINT_PATHS = [str(RELEASED_DIR / f"rep_{r}" / "model.pt") for r in REPS]
OUTPUT_DIR = Path("results/demo")

RUN_PRETRAINING = False   # True: train a new backbone (300 epochs, GPU recommended)
RUN_FINETUNING = True
RUN_ALL_SEEDS = True

## 1. Setup

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_curve
from torch.utils.data import DataLoader

RELEASE_ROOT = Path.cwd()
assert (RELEASE_ROOT / "utils" / "pipeline.py").is_file(), "Run this notebook from the repository root."
sys.path.insert(0, str(RELEASE_ROOT))

from utils import pipeline as P
from scripts.finetune import finetune_mode

# Paper settings from configs/synpred.yaml; paths come from the cell above.
cfg = P.load_config("configs/synpred.yaml")
cfg["data"].update(data_path=DATA_PATH, rna_data_path=RNA_PATH)
cfg["device"] = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE, SEED = cfg["device"], cfg["pretrain"]["seeds"][0]   # 42
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
P.set_seed(SEED)
print("device:", DEVICE)

## 2. Data

The RNA file stores **genes as rows** and patient IDs in the header. `P.load_rna_table` transposes it once to `patients x genes`. The `Dataset` then mean-imputes, applies `log1p` and z-scores it.

Each split pickle maps `patient_id -> {"pcr": 0/1, "T0_early": path, "T0_late": path}`. Patient IDs must match the RNA header (strings). The table below shows how many patients per split have RNA.

### Patient overlap between splits

Checks that no patient ID, and no image file, appears in more than one split.

In [ ]:
import itertools

splits = P.load_splits(DATA_PATH)
rows = []
for a, b in itertools.combinations(splits, 2):
    shared_ids = set(splits[a]) & set(splits[b])
    images = lambda s: {r[k] for r in splits[s].values() for k in ("T0_early", "T0_late")}
    shared_images = images(a) & images(b)
    rows.append({"splits": f"{a} / {b}", "shared patient IDs": len(shared_ids),
                 "shared images": len(shared_images), "examples": sorted(shared_ids)[:5]})
display(pd.DataFrame(rows).set_index("splits"))
assert all(r["shared patient IDs"] == 0 and r["shared images"] == 0 for r in rows), "Patients leak across splits."
print("No overlap between train, val and test.")

In [ ]:
datasets = P.build_datasets(cfg["data"])
loaders = P.build_loaders(datasets, cfg["data"], SEED)
RNA_DIM = P.rna_dim(datasets)

### Samples from the dataloader

In [ ]:
batch = next(iter(DataLoader(datasets["train_clean"], batch_size=6, shuffle=True,
                             generator=torch.Generator().manual_seed(SEED))))
print({k: (tuple(v.shape) if torch.is_tensor(v) else type(v).__name__) for k, v in batch.items()})

n = len(batch["id"])
fig, axes = plt.subplots(3, n, figsize=(2.6 * n, 8))
for i in range(n):
    clean = batch["T0"][i]
    augmented = P.IMAGE_AUGMENTATION(clean)
    for row, (img, label) in enumerate([(clean[0], "early"), (clean[1], "late"), (augmented[0], "early, augmented")]):
        axes[row, i].imshow(np.rot90(np.asarray(img)), cmap="gray")
        axes[row, i].set_xticks([]); axes[row, i].set_yticks([])
        if i == 0:
            axes[row, i].set_ylabel(label)
    rna_flag = "no RNA" if bool(batch["rna_missing"][i]) else "RNA"
    axes[0, i].set_title(f"{batch['id'][i]}\npCR={int(batch['pcr'][i])} | {rna_flag}", fontsize=9)
plt.suptitle("T0 maximum-intensity projections (early / late subtraction)")
plt.tight_layout(); plt.show()

present = ~batch["rna_missing"]
if present.any():
    plt.figure(figsize=(8, 2.5))
    plt.imshow(batch["rna"][present][:, :200].numpy(), aspect="auto", cmap="coolwarm", vmin=-3, vmax=3)
    plt.colorbar(label="z-score"); plt.xlabel("gene (first 200)"); plt.ylabel("patient")
    plt.title("Preprocessed RNA input"); plt.show()

## 3. Backbone: pretrain or load

The latent of each modality is split into `z_uni` (16), `z_fusion` (16, combined across modalities by MoE) and `z_inst` (224).

`RUN_PRETRAINING = True` runs the same code as `python scripts/pretrain.py --seeds 42` (one run) and adds the selected and the last-epoch checkpoint to `CHECKPOINT_PATHS` and `LAST_CHECKPOINT_PATHS`. Without `--seeds`, the script runs every seed in `pretrain.seeds`. Pretraining needs a GPU; on the full data, run it from the command line.

In [ ]:
if RUN_PRETRAINING:
    import subprocess
    subprocess.run([sys.executable, "scripts/pretrain.py", "--data-path", DATA_PATH,
                    "--rna-data-path", RNA_PATH, "--output-dir", str(OUTPUT_DIR),
                    "--device", DEVICE, "--seeds", str(SEED)], check=True)
    CHECKPOINT_PATHS = [str(OUTPUT_DIR / f"seed_{SEED}" / "model_best_auc.pt")] + list(CHECKPOINT_PATHS)
    LAST_CHECKPOINT_PATHS = [str(OUTPUT_DIR / f"seed_{SEED}" / "model_last.pt")] + list(LAST_CHECKPOINT_PATHS)

assert CHECKPOINT_PATHS and LAST_CHECKPOINT_PATHS, "Set the checkpoint paths or RUN_PRETRAINING = True."
MODEL_CFG = P.model_config_for_checkpoint(cfg, CHECKPOINT_PATHS[0])
model = P.load_state(P.build_model(MODEL_CFG, RNA_DIM, device=DEVICE), CHECKPOINT_PATHS[0], DEVICE).eval()
model_last = P.load_state(P.build_model(P.model_config_for_checkpoint(cfg, LAST_CHECKPOINT_PATHS[0]), RNA_DIM,
                                        device=DEVICE), LAST_CHECKPOINT_PATHS[0], DEVICE).eval()
print("loaded", CHECKPOINT_PATHS[0], "and", LAST_CHECKPOINT_PATHS[0])

# Training curves, if the checkpoint folder has a history.csv (written by scripts/pretrain.py).
history_path = Path(CHECKPOINT_PATHS[0]).parent / "history.csv"
if history_path.is_file():
    hist = pd.read_csv(history_path)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    ax[0].plot(hist["epoch"], hist["loss_total"], label="train")
    if "val_loss_total" in hist:
        ax[0].plot(hist["epoch"], hist["val_loss_total"], label="val")
    ax[0].set(xlabel="epoch", ylabel="total loss", title="Pretraining loss"); ax[0].legend()
    ax[1].plot(hist["epoch"], hist["val_auroc_mm"])
    selectable = hist[hist["epoch"] >= min(cfg["pretrain"]["min_selection_epoch"], hist["epoch"].max())]
    ax[1].axvline(selectable.loc[selectable["val_auroc_mm"].idxmax(), "epoch"],
                  color="gray", ls="--", label="selected checkpoint")
    ax[1].set(xlabel="epoch", ylabel="AUROC", title="Val AUROC (mm, linear probe)"); ax[1].legend()
    plt.tight_layout(); plt.show()
else:
    print(f"No history.csv next to {CHECKPOINT_PATHS[0]}; skipping the training curves.")

### Reconstructions (sanity check)

Rows: input, reconstruction from the selected checkpoint, reconstruction from the last-epoch checkpoint. The image decoder gets most of its spatial detail from the encoder skip connection at 1/8 resolution, so reconstructions are blurry by design. The selected checkpoint (often an early epoch) can still show a saturated, striped blob at the tumour.

In [ ]:
with torch.no_grad():
    b = next(iter(DataLoader(datasets["test"], batch_size=4, shuffle=False)))
    recs = {name: m(img_x=b["T0"].to(DEVICE), eval=True)["img_rec"].cpu()
            for name, m in [("selected", model), ("last epoch", model_last)]}
rows = [("input (early)", b["T0"])] + [(f"reconstruction, {k}", v) for k, v in recs.items()]
fig, axes = plt.subplots(len(rows), 4, figsize=(10, 2.6 * len(rows)))
for r, (label, imgs) in enumerate(rows):
    for i in range(4):
        axes[r, i].imshow(np.rot90(imgs[i, 0].numpy()), cmap="gray", vmin=0, vmax=1)
        axes[r, i].set_xticks([]); axes[r, i].set_yticks([])
    axes[r, 0].set_ylabel(label, fontsize=9)
for i in range(4):
    axes[0, i].set_title(b["id"][i], fontsize=9)
plt.tight_layout(); plt.show()

## 4. Frozen evaluation: linear probe

Same as `python scripts/evaluate.py --checkpoint ...`. A logistic regression is fitted on **multimodal** train embeddings. It is then evaluated on IMG-only (all test patients), RNA-only and multimodal (patients with RNA) embeddings, with no retraining.

In [ ]:
probe_dir = OUTPUT_DIR / "frozen_eval"
probe_results = P.frozen_evaluation(model, loaders, DEVICE, probe_dir)
probe_val = {m: pd.read_csv(probe_dir / f"predictions_val_{m}.csv") for m in probe_results["val"]}
probe_test = {m: pd.read_csv(probe_dir / f"predictions_test_{m}.csv") for m in probe_results["test"]}
display(pd.DataFrame(probe_results["test"]).T.style.format("{:.3f}"))

### Last-epoch checkpoint

The same linear probe on the backbone from the final pretraining epoch, so the effect of selecting the checkpoint by validation AUROC can be seen. Outputs go to `frozen_eval_last/`.

In [ ]:
probe_last = P.frozen_evaluation(model_last, loaders, DEVICE, OUTPUT_DIR / "frozen_eval_last")
compare = pd.concat({"selected (val AUROC)": pd.DataFrame(probe_results["test"]).T[["auroc", "prauc"]],
                     "last epoch": pd.DataFrame(probe_last["test"]).T[["auroc", "prauc"]]}, axis=1)
display(compare.astype(float).style.format("{:.3f}"))

## 5. Fine-tuning

Same as `python scripts/finetune.py --checkpoint ...`. One MLP head is trained per mode on multimodal embeddings, with the backbone frozen (`finetune.freeze_backbone` in the config). Heads are selected by validation loss. The backbone is the checkpoint selected by validation AUROC (`finetune.backbone: best_auc`, the default); set it to `last` to fine-tune the last-epoch checkpoint.

In [ ]:
if RUN_FINETUNING:
    ft_cfg = cfg["finetune"]
    FT_CHECKPOINT = CHECKPOINT_PATHS[0] if ft_cfg["backbone"] == "best_auc" else LAST_CHECKPOINT_PATHS[0]
    print("fine-tuning on", FT_CHECKPOINT)
    ft_val, ft_test = {}, {}
    for mode in ft_cfg["modes"]:
        ft = finetune_mode(mode, FT_CHECKPOINT, MODEL_CFG, ft_cfg, datasets, loaders, DEVICE)
        ft_val[mode], _ = P.predict_head(ft, loaders["val"], mode, DEVICE, ft_cfg["mm_paired_only"])
        ft_test[mode], _ = P.predict_head(ft, loaders["test"], mode, DEVICE, ft_cfg["mm_paired_only"])
    display(pd.DataFrame({m: P.metrics(d) for m, d in ft_test.items()}).T.style.format("{:.3f}"))

## 6. Predictions on test images

The predicted label uses a threshold chosen on the **validation** set (Youden's J), because pCR is the minority class. Green titles are correct predictions and red titles are errors. The default shows image-only predictions, which cover every test patient.

In [ ]:
SOURCE = "fine-tuned" if RUN_FINETUNING else "linear probe"
pred_val = ft_val if RUN_FINETUNING else probe_val
pred_test = ft_test if RUN_FINETUNING else probe_test
SHOW_MODE = "img"

fpr, tpr, thr_all = roc_curve(pred_val[SHOW_MODE].y_true, pred_val[SHOW_MODE].prob)
thr = float(thr_all[np.argmax(tpr - fpr)])
df = pred_test[SHOW_MODE].assign(y_pred=lambda d: (d.prob >= thr).astype(int))
test_items = {item["id"]: item for item in datasets["test"].data_list}

# Up to 3 examples of each outcome (TN, FP, FN, TP).
picks = pd.concat([g.sample(min(len(g), 3), random_state=SEED) for _, g in df.groupby(["y_true", "y_pred"])])
fig, axes = plt.subplots(1, len(picks), figsize=(2.6 * len(picks), 3.2))
for ax, (_, r) in zip(np.atleast_1d(axes), picks.iterrows()):
    ax.imshow(np.rot90(test_items[r.id]["T0"][0].numpy()), cmap="gray")
    ax.set_title(f"{r.id}\ntrue {r.y_true} | pred {r.y_pred}", fontsize=8,
                 color="green" if r.y_true == r.y_pred else "red")
    ax.axis("off")
plt.suptitle(f"{SOURCE}, {SHOW_MODE} input, threshold {thr:.2f} (1 = pCR)")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
for m, d in pred_test.items():
    f, t, _ = roc_curve(d.y_true, d.prob)
    ax.plot(f, t, label=f"{m} (AUROC {P.metrics(d)['auroc']:.3f}, n={len(d)})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.legend(); ax.set_title(f"Test ROC ({SOURCE})")
plt.tight_layout(); plt.show()

## 7. All pretraining repetitions

Linear probe on the **last-epoch** checkpoint of each repetition (`LAST_CHECKPOINT_PATHS`): mean and standard deviation across repetitions, plus the AUROC of the ensemble (mean predicted probability). From the command line: `scripts/evaluate.py` per checkpoint, then `scripts/summarize.py`.

In [ ]:
if RUN_ALL_SEEDS and len(LAST_CHECKPOINT_PATHS) > 1:
    rows, ensemble = [], {}
    for ckpt in LAST_CHECKPOINT_PATHS:
        m_k = P.load_state(P.build_model(P.model_config_for_checkpoint(cfg, ckpt), RNA_DIM, device=DEVICE),
                           ckpt, DEVICE).eval()
        emb_train = P.extract_embeddings(m_k, loaders["train_clean"], DEVICE)
        res = P.linear_probe(emb_train, P.extract_embeddings(m_k, loaders["test"], DEVICE))
        rows.append({"checkpoint": Path(ckpt).parent.name, **{m: P.metrics(d)["auroc"] for m, d in res.items()}})
        for m, d in res.items():
            ensemble.setdefault(m, []).append(d)
    per_seed = pd.DataFrame(rows).set_index("checkpoint")
    display(per_seed.style.format("{:.3f}"))
    agg = per_seed.agg(["mean", "std"]).T
    agg["ensemble"] = [P.metrics(pd.concat(ensemble[m]).groupby("id", as_index=False)
                                 .agg(y_true=("y_true", "first"), prob=("prob", "mean")))["auroc"]
                       for m in agg.index]
    display(agg.style.format("{:.3f}"))